In [1]:
!pip install torch torchvision torchaudio
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.9 MB/s eta 0:00:00


In [2]:
from torch_geometric.datasets import QM9

dataset = QM9(root='qm9_data')

Extracting qm9_data/raw/qm9_v3.zip
Processing...
Using a pre-processed version of the dataset. Please install 'rdkit' to alternatively process the raw data.
Done!


In [3]:
print(dataset.num_features)

11


In [4]:
# =============================================================================
# E(n) Equivariant Graph Neural Networks (EGNN) — QM9 Molecular Property Prediction
# Paper: https://proceedings.mlr.press/v139/satorras21a/satorras21a.pdf
# Original repo: https://github.com/vgsatorras/egnn
#
# ── DATASET ──────────────────────────────────────────────────────────────────
# Uses the zaharch Kaggle dataset:
#   https://www.kaggle.com/datasets/zaharch/quantum-machine-9-aka-qm9
#
# After adding the dataset to your notebook the files are at:
#   /kaggle/input/quantum-machine-9-aka-qm9/dsgdb9nsd.xyz/dsgdb9nsd_000001.xyz
#                                                         dsgdb9nsd_000002.xyz
#                                                         ... (133,885 files)
#
# Each .xyz file follows the QM9 format:
#   Line 0  : number of atoms (na)
#   Line 1  : "gdb <index>\t<mu>\t<alpha>\t<homo>\t...<zpve>"  (17 properties)
#   Lines 2 to na+1: "<element>\t<x>\t<y>\t<z>\t<Mulliken_charge>"
#   Line na+2 : harmonic vibrational frequencies
#   Line na+3 : SMILES
#   Line na+4 : InChI
#
# QM9 property order on line 1 (after 'gdb <idx>'):
#   [0]tag  [1]idx  [2]A  [3]B  [4]C  [5]mu  [6]alpha  [7]homo  [8]lumo
#   [9]gap  [10]r2  [11]zpve  [12]U0  [13]U  [14]H  [15]G  [16]Cv
# =============================================================================

import os, json, time, random, glob
import numpy as np
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


# =============================================================================
# SECTION 1 — EGNN MODEL
# Faithfully extracted from models/egnn_clean/egnn_clean.py and qm9/models.py
# =============================================================================

def unsorted_segment_sum(data, segment_ids, num_segments):
    """Scatter-add: sums data rows into rows indexed by segment_ids."""
    result_shape = (num_segments, data.size(1))
    result = data.new_full(result_shape, 0)
    segment_ids = segment_ids.unsqueeze(-1).expand(-1, data.size(1))
    result.scatter_add_(0, segment_ids, data)
    return result


def unsorted_segment_mean(data, segment_ids, num_segments):
    """Scatter-mean: averages data rows into rows indexed by segment_ids."""
    result_shape = (num_segments, data.size(1))
    segment_ids_exp = segment_ids.unsqueeze(-1).expand(-1, data.size(1))
    result = data.new_full(result_shape, 0)
    count  = data.new_full(result_shape, 0)
    result.scatter_add_(0, segment_ids_exp, data)
    count.scatter_add_(0, segment_ids_exp, torch.ones_like(data))
    return result / count.clamp(min=1)


class E_GCL(nn.Module):
    """
    E(n) Equivariant Graph Convolutional Layer.

    Updates node features h and coordinates x jointly while preserving
    E(n) equivariance (rotations, translations, reflections, permutations).
    """

    def __init__(self, input_nf, output_nf, hidden_nf,
                 edges_in_d=0, act_fn=nn.SiLU(),
                 residual=True, attention=False,
                 normalize=False, coords_agg='mean', tanh=False):
        super().__init__()
        self.residual   = residual
        self.attention  = attention
        self.normalize  = normalize
        self.coords_agg = coords_agg
        self.tanh       = tanh
        self.epsilon    = 1e-8

        input_edge     = input_nf * 2
        edge_coords_nf = 1   # squared distance

        # Edge MLP: φ_e
        self.edge_mlp = nn.Sequential(
            nn.Linear(input_edge + edge_coords_nf + edges_in_d, hidden_nf),
            act_fn,
            nn.Linear(hidden_nf, hidden_nf),
            act_fn,
        )

        # Node MLP: φ_h
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_nf + input_nf, hidden_nf),
            act_fn,
            nn.Linear(hidden_nf, output_nf),
        )

        # Coordinate MLP: φ_x  (small xavier init for stability)
        coord_mlp = [nn.Linear(hidden_nf, hidden_nf), act_fn]
        last_layer = nn.Linear(hidden_nf, 1, bias=False)
        nn.init.xavier_uniform_(last_layer.weight, gain=0.001)
        coord_mlp.append(last_layer)
        if self.tanh:
            coord_mlp.append(nn.Tanh())
        self.coord_mlp = nn.Sequential(*coord_mlp)

        if self.attention:
            self.att_mlp = nn.Sequential(nn.Linear(hidden_nf, 1), nn.Sigmoid())

    def coord2radial(self, edge_index, coord):
        row, col = edge_index
        coord_diff = coord[row] - coord[col]
        radial = torch.sum(coord_diff ** 2, dim=1, keepdim=True)
        if self.normalize:
            norm = torch.sqrt(radial).detach() + self.epsilon
            coord_diff = coord_diff / norm
        return radial, coord_diff

    def edge_model(self, source, target, radial, edge_attr):
        if edge_attr is None:
            out = torch.cat([source, target, radial], dim=1)
        else:
            out = torch.cat([source, target, radial, edge_attr], dim=1)
        out = self.edge_mlp(out)
        if self.attention:
            out = out * self.att_mlp(out)
        return out

    def coord_model(self, coord, edge_index, coord_diff, edge_feat):
        row, _ = edge_index
        trans = coord_diff * self.coord_mlp(edge_feat)
        if self.coords_agg == 'sum':
            agg = unsorted_segment_sum(trans, row, num_segments=coord.size(0))
        elif self.coords_agg == 'mean':
            agg = unsorted_segment_mean(trans, row, num_segments=coord.size(0))
        else:
            raise ValueError(f"Unknown coords_agg: {self.coords_agg}")
        return coord + agg

    def node_model(self, x, edge_index, edge_attr, node_attr):
        row, _ = edge_index
        agg = unsorted_segment_sum(edge_attr, row, num_segments=x.size(0))
        if node_attr is not None:
            agg = torch.cat([x, agg, node_attr], dim=1)
        else:
            agg = torch.cat([x, agg], dim=1)
        out = self.node_mlp(agg)
        if self.residual:
            out = x + out
        return out, agg

    def forward(self, h, edge_index, coord, edge_attr=None, node_attr=None):
        row, col = edge_index
        radial, coord_diff = self.coord2radial(edge_index, coord)
        edge_feat           = self.edge_model(h[row], h[col], radial, edge_attr)
        coord               = self.coord_model(coord, edge_index, coord_diff, edge_feat)
        h, _                = self.node_model(h, edge_index, edge_feat, node_attr)
        return h, coord, edge_attr


class EGNN(nn.Module):
    """Clean EGNN (egnn_clean.py) — for use outside QM9."""

    def __init__(self, in_node_nf, hidden_nf, out_node_nf,
                 in_edge_nf=0, device='cpu', act_fn=nn.SiLU(),
                 n_layers=4, residual=True, attention=False,
                 normalize=False, tanh=False):
        super().__init__()
        self.hidden_nf = hidden_nf
        self.device    = device
        self.n_layers  = n_layers
        self.embedding_in  = nn.Linear(in_node_nf, hidden_nf)
        self.embedding_out = nn.Linear(hidden_nf, out_node_nf)
        for i in range(n_layers):
            self.add_module(f"gcl_{i}", E_GCL(
                hidden_nf, hidden_nf, hidden_nf, edges_in_d=in_edge_nf,
                act_fn=act_fn, residual=residual, attention=attention,
                normalize=normalize, tanh=tanh,
            ))
        self.to(device)

    def forward(self, h, x, edges, edge_attr=None):
        h = self.embedding_in(h)
        for i in range(self.n_layers):
            h, x, _ = self._modules[f"gcl_{i}"](h, edges, x, edge_attr=edge_attr)
        h = self.embedding_out(h)
        return h, x


class EGNN_QM9(nn.Module):
    """
    EGNN for QM9 molecular property prediction.
    Adds node/edge masking and a graph-level mean-pool + MLP readout.
    """

    def __init__(self, in_node_nf, in_edge_nf, hidden_nf,
                 device='cpu', act_fn=nn.SiLU(), n_layers=7,
                 coords_weight=1.0, attention=True, node_attr=1):
        super().__init__()
        self.hidden_nf     = hidden_nf
        self.device        = device
        self.n_layers      = n_layers
        self.coords_weight = coords_weight

        self.embedding = nn.Linear(in_node_nf, hidden_nf)
        for i in range(n_layers):
            self.add_module(f"gcl_{i}", E_GCL(
                hidden_nf, hidden_nf, hidden_nf, edges_in_d=in_edge_nf,
                act_fn=act_fn, residual=True, attention=attention,
                normalize=False, tanh=False,
            ))
        self.node_dec = nn.Sequential(
            nn.Linear(hidden_nf, hidden_nf), act_fn,
            nn.Linear(hidden_nf, hidden_nf),
        )
        self.graph_dec = nn.Sequential(
            nn.Linear(hidden_nf, hidden_nf), act_fn,
            nn.Linear(hidden_nf, 1),
        )
        self.to(device)

    def forward(self, h0, x, edges, edge_attr=None,
                node_mask=None, edge_mask=None, n_nodes=None):
        h = self.embedding(h0)
        for i in range(self.n_layers):
            h, x, _ = self._modules[f"gcl_{i}"](h, edges, x, edge_attr=edge_attr)

        h = self.node_dec(h)
        if node_mask is not None:
            h = h * node_mask

        if n_nodes is not None:
            batch_size = h.size(0) // n_nodes
            h = h.view(batch_size, n_nodes, -1)
            if node_mask is not None:
                mask = node_mask.view(batch_size, n_nodes, 1)
                h = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            else:
                h = h.mean(dim=1)
        else:
            h = h.mean(dim=0, keepdim=True)

        return self.graph_dec(h).squeeze(-1)


# ── Edge utilities ────────────────────────────────────────────────────────────

def get_edges(n_nodes):
    rows, cols = [], []
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                rows.append(i)
                cols.append(j)
    return [rows, cols]


def get_edges_batch(n_nodes, batch_size):
    edges = get_edges(n_nodes)
    edge_attr = torch.ones(len(edges[0]) * batch_size, 1)
    edges = [torch.LongTensor(edges[0]), torch.LongTensor(edges[1])]
    if batch_size == 1:
        return edges, edge_attr
    rows, cols = [], []
    for i in range(batch_size):
        rows.append(edges[0] + n_nodes * i)
        cols.append(edges[1] + n_nodes * i)
    return [torch.cat(rows), torch.cat(cols)], edge_attr


_adj_cache = {}

def get_adj_matrix(n_nodes, batch_size, device):
    key = (n_nodes, batch_size, str(device))
    if key not in _adj_cache:
        edges, _ = get_edges_batch(n_nodes, batch_size)
        _adj_cache[key] = [edges[0].to(device), edges[1].to(device)]
    return _adj_cache[key]


# =============================================================================
# SECTION 2 — QM9 XYZ PARSER
#
# The zaharch Kaggle dataset (quantum-machine-9-aka-qm9) is a folder of
# 133,885 individual xyz files:
#   /kaggle/input/quantum-machine-9-aka-qm9/dsgdb9nsd.xyz/dsgdb9nsd_000001.xyz
#
# XYZ file format (one molecule per file):
#   line 0        : na  (number of atoms, integer)
#   line 1        : "gdb <idx>\t<A>\t<B>\t<C>\t<mu>\t<alpha>\t<homo>\t<lumo>
#                         \t<gap>\t<r2>\t<zpve>\t<U0>\t<U>\t<H>\t<G>\t<Cv>"
#   lines 2..na+1 : "<element>\t<x>\t<y>\t<z>\t<Mulliken_charge>"
#   line na+2     : vibrational frequencies (ignored)
#   line na+3     : SMILES (ignored)
#   line na+4     : InChI  (ignored)
#
# Property index on line 1 (after splitting on whitespace, skipping 'gdb' and idx):
#   0:A  1:B  2:C  3:mu  4:alpha  5:homo  6:lumo  7:gap
#   8:r2 9:zpve 10:U0 11:U 12:H 13:G 14:Cv
# =============================================================================

# Map from our property name → column index in the scalar properties list
_PROP_COL = {
    'mu'   : 3,
    'alpha': 4,
    'homo' : 5,
    'lumo' : 6,
    'gap'  : 7,
    'r2'   : 8,
    'zpve' : 9,
    'U0'   : 10,
    'U'    : 11,
    'H'    : 12,
    'G'    : 13,
    'Cv'   : 14,
}

QM9_PROPERTIES = list(_PROP_COL.keys())

ATOM_ENCODER   = {'H': 0, 'C': 1, 'N': 2, 'O': 3, 'F': 4}
NUM_ATOM_TYPES = len(ATOM_ENCODER)

_SYM_TO_Z = {'H': 1, 'C': 6, 'N': 7, 'O': 8, 'F': 9,
             'S': 16, 'Cl': 17, 'Br': 35, 'I': 53}


def _parse_float(s):
    """Handle QM9's non-standard float notation: 2.0153*^-6 → 2.0153e-6."""
    return float(s.replace('*^', 'e'))


def parse_xyz_file(path):
    """
    Parse a single QM9 xyz file.

    Returns a dict with keys:
        n_atoms   : int
        positions : (n_atoms, 3) float32
        charges   : (n_atoms,)  float32  (nuclear charges, not Mulliken)
        mu, alpha, homo, lumo, gap, r2, zpve, U0, U, H, G, Cv : float
    Returns None if the file is malformed.
    """
    try:
        with open(path, 'r') as f:
            lines = f.readlines()

        na = int(lines[0].strip())

        # Line 1: scalar properties
        # Format: "gdb <idx>  <A>  <B>  <C>  <mu>  ..."
        parts = lines[1].strip().split()
        # parts[0]='gdb', parts[1]=idx, parts[2..] = A B C mu alpha homo ...
        scalars = [_parse_float(p) for p in parts[2:]]

        mol = {'n_atoms': na}
        for name, col in _PROP_COL.items():
            mol[name] = scalars[col]

        positions = []
        charges   = []
        for i in range(na):
            toks = lines[2 + i].strip().split()
            sym  = toks[0]
            x, y, z = _parse_float(toks[1]), _parse_float(toks[2]), _parse_float(toks[3])
            positions.append([x, y, z])
            charges.append(float(_SYM_TO_Z.get(sym, 6)))   # default to C

        mol['positions'] = np.array(positions, dtype=np.float32)
        mol['charges']   = np.array(charges,   dtype=np.float32)
        return mol

    except Exception:
        return None


def load_qm9_xyz(xyz_dir, max_samples=None, verbose=True):
    """
    Load QM9 from the zaharch Kaggle dataset directory of .xyz files.

    Args:
        xyz_dir     : path to the dsgdb9nsd.xyz/ folder
                      (e.g. '/kaggle/input/quantum-machine-9-aka-qm9/dsgdb9nsd.xyz')
        max_samples : if set, load only the first N molecules
        verbose     : print progress every 10k files

    Returns list of molecule dicts.
    """
    if not os.path.isdir(xyz_dir):
        raise FileNotFoundError(
            f"XYZ directory not found: {xyz_dir}\n"
            "Make sure you have added the dataset\n"
            "  zaharch/quantum-machine-9-aka-qm9\n"
            "to your Kaggle notebook via Add Data."
        )

    files = sorted(glob.glob(os.path.join(xyz_dir, '*.xyz')))
    if not files:
        raise FileNotFoundError(f"No .xyz files found inside {xyz_dir}")

    if max_samples:
        files = files[:max_samples]

    data_list = []
    n_skipped = 0
    for i, fpath in enumerate(files):
        mol = parse_xyz_file(fpath)
        if mol is None:
            n_skipped += 1
            continue
        data_list.append(mol)
        if verbose and (i + 1) % 10000 == 0:
            print(f"  Loaded {i+1}/{len(files)} molecules …")

    print(f"Loaded {len(data_list)} molecules  (skipped {n_skipped} malformed files).")
    return data_list


def find_xyz_dir():
    """
    Auto-detect the zaharch QM9 xyz directory on Kaggle.
    Tries several known locations and returns the first that exists.
    """
    candidates = [
        # zaharch/quantum-machine-9-aka-qm9  (most common slug)
        '/kaggle/input/quantum-machine-9-aka-qm9/dsgdb9nsd.xyz',
        # older slug variant
        '/kaggle/input/qm9-quantum-machine-9/dsgdb9nsd.xyz',
        # generic fallback: search under /kaggle/input
        *glob.glob('/kaggle/input/*/dsgdb9nsd.xyz'),
    ]
    for c in candidates:
        if os.path.isdir(c):
            n = len(glob.glob(os.path.join(c, '*.xyz')))
            print(f"Found QM9 xyz directory: {c}  ({n} files)")
            return c
    return None


# =============================================================================
# SECTION 3 — PYTORCH DATASET
# =============================================================================

class QM9Dataset(Dataset):
    """
    QM9 dataset for EGNN training.

    Pads each molecule to max_atoms with zero-masks.
    Returns:
        positions  : (max_atoms, 3)
        one_hot    : (max_atoms, NUM_ATOM_TYPES)
        charges    : (max_atoms, 1)
        atom_mask  : (max_atoms, 1)
        edge_mask  : (max_atoms*max_atoms,)
        <property> : scalar float tensor
    """

    def __init__(self, data_list, property='homo', max_atoms=29):
        self.data      = data_list
        self.property  = property
        self.max_atoms = max_atoms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        mol = self.data[idx]
        n   = mol['n_atoms']
        N   = self.max_atoms

        pos = np.zeros((N, 3), dtype=np.float32)
        pos[:n] = mol['positions'][:n]

        one_hot = np.zeros((N, NUM_ATOM_TYPES), dtype=np.float32)
        for i, z in enumerate(mol['charges'][:n]):
            sym = {1:'H', 6:'C', 7:'N', 8:'O', 9:'F'}.get(int(z), 'C')
            if sym in ATOM_ENCODER:
                one_hot[i, ATOM_ENCODER[sym]] = 1.0

        charges = np.zeros((N, 1), dtype=np.float32)
        charges[:n, 0] = mol['charges'][:n]

        atom_mask = np.zeros((N, 1), dtype=np.float32)
        atom_mask[:n, 0] = 1.0

        edge_mask = np.zeros((N * N,), dtype=np.float32)
        for i in range(n):
            for j in range(n):
                if i != j:
                    edge_mask[i * N + j] = 1.0

        return {
            'positions' : torch.FloatTensor(pos),
            'one_hot'   : torch.FloatTensor(one_hot),
            'charges'   : torch.FloatTensor(charges),
            'atom_mask' : torch.FloatTensor(atom_mask),
            'edge_mask' : torch.FloatTensor(edge_mask),
            self.property: torch.FloatTensor([mol[self.property]]),
        }


def split_dataset(data_list, train_size=110000, val_size=10000, test_size=10000, seed=42):
    """Standard QM9 train/val/test split (same as original EGNN paper)."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(data_list))
    train_idx = idx[:train_size]
    val_idx   = idx[train_size:train_size + val_size]
    test_idx  = idx[train_size + val_size:train_size + val_size + test_size]
    return (
        [data_list[i] for i in train_idx],
        [data_list[i] for i in val_idx],
        [data_list[i] for i in test_idx],
    )


# ── Synthetic fallback (only used when no real data is available) ─────────────

def _synthetic_qm9(n=500):
    """Synthetic molecules for smoke-testing without real files."""
    data_list = []
    for _ in range(n):
        na = np.random.randint(3, 10)
        mol = {
            'n_atoms'   : na,
            'positions' : np.random.randn(na, 3).astype(np.float32),
            'charges'   : np.random.choice([1, 6, 7, 8], size=na).astype(np.float32),
        }
        for p in QM9_PROPERTIES:
            mol[p] = float(np.random.randn())
        data_list.append(mol)
    return data_list


# =============================================================================
# SECTION 4 — NODE FEATURE PREPROCESSING  (mirrors qm9/utils.py)
# =============================================================================

def preprocess_input(one_hot, charges, charge_power, charge_scale, device):
    """
    Build node features: [one_hot | (z/scale)^1 | ... | (z/scale)^charge_power]
    in_node_nf = NUM_ATOM_TYPES + charge_power
    """
    if one_hot.dim() == 3:
        B, N, F = one_hot.shape
        one_hot = one_hot.view(B * N, F)

    if charges.dim() == 3:
        charges = charges.view(-1, 1)

    charge_tensor = (charges / charge_scale).pow(
        torch.arange(1, charge_power + 1, device=device, dtype=torch.float32)
    )
    return torch.cat([one_hot, charge_tensor], dim=1)


def compute_mean_mad(dataloaders, prop_name):
    """Compute mean and MAD of target property on training set."""
    values = []
    for batch in dataloaders['train']:
        values.append(batch[prop_name])
    values = torch.cat(values, dim=0).float()
    mean = values.mean().item()
    mad  = (values - mean).abs().mean().item()
    print(f"Property '{prop_name}':  mean={mean:.6f},  MAD={mad:.6f}")
    return mean, mad


# =============================================================================
# SECTION 5 — TRAINING & EVALUATION LOOPS
# =============================================================================

def train_epoch(model, loader, optimizer, loss_fn, mean, mad,
                prop, charge_power, charge_scale, n_nodes, device,
                dtype=torch.float32):
    model.train()
    total_loss = 0.0
    n_samples  = 0

    for batch in loader:
        optimizer.zero_grad()
        B = batch['positions'].size(0)

        atom_positions = batch['positions'].view(B * n_nodes, -1).to(device, dtype)
        atom_mask      = batch['atom_mask'].view(B * n_nodes, -1).to(device, dtype)
        one_hot        = batch['one_hot'].to(device, dtype)
        charges        = batch['charges'].to(device, dtype)
        label          = batch[prop].to(device, dtype).squeeze(-1)

        nodes = preprocess_input(one_hot, charges, charge_power, charge_scale, device)
        nodes = nodes.view(B * n_nodes, -1)
        edges = get_adj_matrix(n_nodes, B, device)

        pred = model(h0=nodes, x=atom_positions, edges=edges, edge_attr=None,
                     node_mask=atom_mask, n_nodes=n_nodes)

        loss = loss_fn(pred, (label - mean) / mad)   # normalised MAE
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * B
        n_samples  += B

    return total_loss / n_samples


@torch.no_grad()
def eval_epoch(model, loader, loss_fn, mean, mad,
               prop, charge_power, charge_scale, n_nodes, device,
               dtype=torch.float32):
    model.eval()
    total_mae = 0.0
    n_samples = 0

    for batch in loader:
        B = batch['positions'].size(0)

        atom_positions = batch['positions'].view(B * n_nodes, -1).to(device, dtype)
        atom_mask      = batch['atom_mask'].view(B * n_nodes, -1).to(device, dtype)
        one_hot        = batch['one_hot'].to(device, dtype)
        charges        = batch['charges'].to(device, dtype)
        label          = batch[prop].to(device, dtype).squeeze(-1)

        nodes = preprocess_input(one_hot, charges, charge_power, charge_scale, device)
        nodes = nodes.view(B * n_nodes, -1)
        edges = get_adj_matrix(n_nodes, B, device)

        pred = model(h0=nodes, x=atom_positions, edges=edges, edge_attr=None,
                     node_mask=atom_mask, n_nodes=n_nodes)

        mae = loss_fn(mad * pred + mean, label)   # de-normalised MAE
        total_mae += mae.item() * B
        n_samples += B

    return total_mae / n_samples


# =============================================================================
# SECTION 6 — MAIN TRAINING ENTRY POINT
# =============================================================================

def main():
    # ── Config ────────────────────────────────────────────────────────────────
    # Property to predict — choose one of:
    #   alpha | gap | homo | lumo | mu | Cv | G | H | r2 | U | U0 | zpve
    PROPERTY     = 'homo'
    LR           = 1e-3      # 5e-4 for most props; 1e-3 for gap/homo/lumo
    BATCH_SIZE   = 96
    EPOCHS       = 1000      # reduce to 50 for a quick test
    N_LAYERS     = 7
    HIDDEN_NF    = 128
    ATTENTION    = 1
    CHARGE_POWER = 2         # in_node_nf = NUM_ATOM_TYPES + CHARGE_POWER = 7
    MAX_ATOMS    = 29        # QM9 molecules have at most 29 atoms
    NUM_WORKERS  = 2
    WEIGHT_DECAY = 1e-16

    # ── Locate dataset ────────────────────────────────────────────────────────
    xyz_dir = find_xyz_dir()

    if xyz_dir is not None:
        # ── Real QM9 ──────────────────────────────────────────────────────────
        print(f"\nLoading QM9 xyz files from:\n  {xyz_dir}\n")
        all_data = load_qm9_xyz(xyz_dir)

        n_total = len(all_data)
        if n_total >= 130000:
            _tr, _va, _te = 110000, 10000, 10000
        else:
            _tr = max(BATCH_SIZE, int(n_total * 0.8))
            _va = max(BATCH_SIZE, int(n_total * 0.1))
            _te = max(BATCH_SIZE, n_total - _tr - _va)

        train_data, val_data, test_data = split_dataset(
            all_data, train_size=_tr, val_size=_va, test_size=_te)

    else:
        # ── Synthetic fallback ────────────────────────────────────────────────
        print(
            "\n⚠  Could not find the QM9 xyz directory.\n"
            "   To use real data, add the Kaggle dataset:\n"
            "     zaharch / quantum-machine-9-aka-qm9\n"
            "   via Add Data in your notebook.\n"
            "   Running on SYNTHETIC data for now.\n"
        )
        all_data   = _synthetic_qm9(500)
        _tr        = max(BATCH_SIZE, int(len(all_data) * 0.8))
        _va        = max(BATCH_SIZE, int(len(all_data) * 0.1))
        _te        = max(BATCH_SIZE, len(all_data) - _tr - _va)
        train_data, val_data, test_data = split_dataset(
            all_data, train_size=_tr, val_size=_va, test_size=_te)

    print(f"Split → train={len(train_data)}  val={len(val_data)}  test={len(test_data)}")

    # ── DataLoaders ───────────────────────────────────────────────────────────
    train_ds = QM9Dataset(train_data, property=PROPERTY, max_atoms=MAX_ATOMS)
    val_ds   = QM9Dataset(val_data,   property=PROPERTY, max_atoms=MAX_ATOMS)
    test_ds  = QM9Dataset(test_data,  property=PROPERTY, max_atoms=MAX_ATOMS)

    _drop_last = len(train_data) > BATCH_SIZE
    dataloaders = {
        'train': DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, drop_last=_drop_last),
        'valid': DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS),
        'test' : DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS),
    }

    # ── Charge scale (max nuclear charge in QM9 is F=9) ──────────────────────
    charge_scale = max(float(max(d['charges'].max() for d in train_data)), 9.0)

    # ── Dataset statistics ────────────────────────────────────────────────────
    mean, mad = compute_mean_mad(dataloaders, PROPERTY)

    # ── Model ─────────────────────────────────────────────────────────────────
    in_node_nf = NUM_ATOM_TYPES + CHARGE_POWER   # 5 + 2 = 7

    model = EGNN_QM9(
        in_node_nf   = in_node_nf,
        in_edge_nf   = 0,
        hidden_nf    = HIDDEN_NF,
        device       = DEVICE,
        act_fn       = nn.SiLU(),
        n_layers     = N_LAYERS,
        coords_weight= 1.0,
        attention    = bool(ATTENTION),
    )
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nModel: EGNN_QM9  ({n_params:,} trainable parameters)\n")

    # ── Optimiser ─────────────────────────────────────────────────────────────
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS)
    loss_l1   = nn.L1Loss()

    # ── Output dirs ───────────────────────────────────────────────────────────
    os.makedirs('/kaggle/working/egnn_logs', exist_ok=True)

    best_val   = float('inf')
    best_test  = float('inf')
    best_epoch = 0
    history    = []

    # ── Training loop ─────────────────────────────────────────────────────────
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()

        train_loss = train_epoch(
            model, dataloaders['train'], optimizer, loss_l1,
            mean, mad, PROPERTY, CHARGE_POWER, charge_scale, MAX_ATOMS, DEVICE,
        )
        scheduler.step()

        val_mae = eval_epoch(
            model, dataloaders['valid'], loss_l1,
            mean, mad, PROPERTY, CHARGE_POWER, charge_scale, MAX_ATOMS, DEVICE,
        )
        test_mae = eval_epoch(
            model, dataloaders['test'], loss_l1,
            mean, mad, PROPERTY, CHARGE_POWER, charge_scale, MAX_ATOMS, DEVICE,
        )

        history.append({'epoch': epoch, 'train_loss': train_loss,
                        'val_mae': val_mae, 'test_mae': test_mae})

        if val_mae < best_val:
            best_val   = val_mae
            best_test  = test_mae
            best_epoch = epoch
            torch.save(model.state_dict(), '/kaggle/working/egnn_best.pt')

        print(f"Epoch {epoch:4d}/{EPOCHS} | "
              f"train={train_loss:.4f} | "
              f"val={val_mae:.4f} | "
              f"test={test_mae:.4f} | "
              f"{time.time()-t0:.1f}s  "
              f"[best val={best_val:.4f} @ ep{best_epoch}]")

    with open('/kaggle/working/egnn_logs/history.json', 'w') as f:
        json.dump(history, f, indent=2)

    print(f"\n=== Training Complete ===")
    print(f"Best val MAE  : {best_val:.4f}")
    print(f"Best test MAE : {best_test:.4f}  (epoch {best_epoch})")
    print("Model saved → /kaggle/working/egnn_best.pt")


# =============================================================================
# SECTION 7 — SMOKE TEST (runs instantly without real data)
# =============================================================================

def smoke_test():
    print("── Smoke test: clean EGNN ──────────────────────────────────────────")
    B, N, F = 4, 5, 7
    h = torch.ones(B * N, F)
    x = torch.randn(B * N, 3)
    edges, edge_attr = get_edges_batch(N, B)
    egnn = EGNN(in_node_nf=F, hidden_nf=32, out_node_nf=1, in_edge_nf=1,
                n_layers=2, attention=True)
    h_out, x_out = egnn(h, x, edges, edge_attr)
    assert h_out.shape == (B * N, 1)
    assert x_out.shape == (B * N, 3)
    print(f"  h_out: {h_out.shape}  x_out: {x_out.shape}")

    print("── Smoke test: QM9 EGNN ─────────────────────────────────────────────")
    model = EGNN_QM9(in_node_nf=7, in_edge_nf=0, hidden_nf=64,
                     device='cpu', n_layers=2, attention=True)
    N2, B2 = 10, 2
    h0   = torch.randn(B2 * N2, 7)
    pos  = torch.randn(B2 * N2, 3)
    mask = torch.ones(B2 * N2, 1)
    edgs = get_adj_matrix(N2, B2, 'cpu')
    pred = model(h0=h0, x=pos, edges=edgs, node_mask=mask, n_nodes=N2)
    assert pred.shape == (B2,), f"Expected ({B2},), got {pred.shape}"
    print(f"  pred: {pred.shape}")

    print("── Smoke test: xyz parser ───────────────────────────────────────────")
    import tempfile
    xyz_content = (
        "5\n"
        "gdb\t1\t157.7118\t157.70997\t157.70699\t0.0\t13.21\t"
        "-0.3877\t0.1171\t0.5048\t35.3641\t0.04967\t-40.47893\t"
        "-40.47645\t-40.47588\t-40.49477\t6.469\n"
        "C\t-0.0126981359\t1.0858041578\t0.0080009958\t-0.535689\n"
        "H\t0.002150416\t-0.0060313176\t0.0019761204\t0.133921\n"
        "H\t1.0117308433\t1.4637511618\t0.0002765748\t0.133922\n"
        "H\t-0.540815069\t1.4475266138\t-0.8766437152\t0.133923\n"
        "H\t-0.5238136345\t1.4379326443\t0.9063972942\t0.133924\n"
        "1352.7085\t1354.1243\n"
        "C\tC([H])([H])[H]\n"
        "InChI=1S/CH4/h1H4\n"
    )
    with tempfile.NamedTemporaryFile(mode='w', suffix='.xyz', delete=False) as f:
        f.write(xyz_content)
        fname = f.name
    mol = parse_xyz_file(fname)
    os.unlink(fname)
    assert mol is not None, "Parser returned None"
    assert mol['n_atoms'] == 5
    assert abs(mol['homo'] - (-0.3877)) < 1e-4
    print(f"  Parsed molecule: n_atoms={mol['n_atoms']}, homo={mol['homo']:.4f}")
    print("  All checks passed ✓\n")


# =============================================================================
if __name__ == "__main__":
    smoke_test()
    main()
else:
    smoke_test()

Using device: cuda
── Smoke test: clean EGNN ──────────────────────────────────────────
  h_out: torch.Size([20, 1])  x_out: torch.Size([20, 3])
── Smoke test: QM9 EGNN ─────────────────────────────────────────────
  pred: torch.Size([2])
── Smoke test: xyz parser ───────────────────────────────────────────
  Parsed molecule: n_atoms=5, homo=-0.3877
  All checks passed ✓


⚠  Could not find the QM9 xyz directory.
   To use real data, add the Kaggle dataset:
     zaharch / quantum-machine-9-aka-qm9
   via Add Data in your notebook.
   Running on SYNTHETIC data for now.

Split → train=400  val=96  test=4
Property 'homo':  mean=-0.075345,  MAD=0.785577

Model: EGNN_QM9  (860,680 trainable parameters)

Epoch    1/1000 | train=1.0049 | val=0.7327 | test=1.0325 | 2.4s  [best val=0.7327 @ ep1]
Epoch    2/1000 | train=1.0021 | val=0.7334 | test=1.0339 | 0.9s  [best val=0.7327 @ ep1]
Epoch    3/1000 | train=1.0048 | val=0.7369 | test=1.0438 | 0.9s  [best val=0.7327 @ ep1]
Epoch    4/1000 | trai

In [5]:
in_node_nf=dataset.num_features